In [0]:
-- ============================================
-- QC — Sessions (Bronze)
-- ============================================

USE CATALOG shoply_analytics;
USE SCHEMA bronze;

-- 1. Preview
SELECT *
FROM sessions
LIMIT 20;

-- 2. Row count
-- 10,000
SELECT COUNT(*) AS row_count
FROM sessions;

-- 3. Duplicate session_id
-- 13 rows
-- 188 nulls
SELECT session_id, COUNT(*) AS cnt
FROM sessions
GROUP BY session_id
HAVING COUNT(*) > 1;

-- 4. Null checks
-- 188 session id 
-- 1686 user id 
-- 4274 campaign id 
-- 340 session start 
-- 1127 session end 
-- 188 device type 
-- 188 country 
-- 5274 utm_campaign
-- 188 utm source 
-- 2565 utm medium
SELECT
  SUM(CASE WHEN session_id IS NULL THEN 1 END) AS null_session_id,
  SUM(CASE WHEN user_id IS NULL THEN 1 END) AS null_user_id,
  SUM(CASE WHEN campaign_id IS NULL THEN 1 END) AS null_campaign_id,
  SUM(CASE WHEN session_start IS NULL THEN 1 END) AS null_session_start,
  SUM(CASE WHEN session_end IS NULL THEN 1 END) AS null_session_end,
  SUM(CASE WHEN device_type IS NULL THEN 1 END) AS null_device_type,
  SUM(CASE WHEN country IS NULL THEN 1 END) AS null_country,
  SUM(CASE WHEN utm_campaign IS NULL THEN 1 END) AS null_utm_campaign,
  SUM(CASE WHEN utm_source IS NULL THEN 1 END) AS null_utm_source,
  SUM(CASE WHEN utm_medium IS NULL THEN 1 END) AS null_utm_medium
FROM sessions;

-- 5. Malformed timestamps 
SELECT *
FROM sessions
WHERE session_start IS NULL AND session_start IS NOT NULL
   OR session_end IS NULL AND session_end IS NOT NULL;

-- 6. Future timestamps
SELECT *
FROM sessions
WHERE session_start > current_timestamp()
   OR session_end > current_timestamp();

-- 7. session_end before session_start
SELECT *
FROM sessions
WHERE session_end < session_start;

-- 8. Unexpected device types
SELECT DISTINCT device_type
FROM sessions;

-- 9. Unwanted spaces in string columns
SELECT *
FROM sessions
WHERE session_id != TRIM(session_id)
   OR user_id != TRIM(user_id)
   OR campaign_id != TRIM(campaign_id)
   OR device_type != TRIM(device_type)
   OR country != TRIM(country)
   OR utm_campaign != TRIM(utm_campaign)
   OR utm_source != TRIM(utm_source)
   OR utm_medium != TRIM(utm_medium);

-- 10. user_id not found in customer_demographics
-- 97
SELECT s.user_id
FROM sessions s
LEFT JOIN customer_demographics d ON s.user_id = d.user_id
WHERE d.user_id IS NULL
  AND s.user_id IS NOT NULL;

-- 11. campaign_id not found in campaigns
-- none 
SELECT s.campaign_id
FROM sessions s
LEFT JOIN campaigns c ON s.campaign_id = c.campaign_id
WHERE c.campaign_id IS NULL
  AND s.campaign_id IS NOT NULL;

-- 12. Column structure
DESCRIBE TABLE sessions;
